In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
#Load cleaned data
resumes_df = pd.read_csv("../data/resumes_cleaned.csv")
jobs_df = pd.read_csv("../data/jobs_cleaned.csv")

#Initialize TF-IDF Vectorizer
vectorizer = TfidfVectorizer()

#Fit on all resumes + jobs text to ensure consistent vocabulary
all_text = pd.concat([resumes_df['clean_text'], jobs_df['clean_text']])
vectorizer.fit(all_text)

#Transform resumes and jobs separately
resume_vectors = vectorizer.transform(resumes_df['clean_text'])
job_vectors = vectorizer.transform(jobs_df['clean_text'])

#Compute Cosine Similarity for each job vs all resumes
similarity_matrix = cosine_similarity(job_vectors, resume_vectors)

#Create rankings
for i, job_id in enumerate(jobs_df['job_id']):
    sim_scores = list(zip(resumes_df['resume_id'], similarity_matrix[i]))
    # Sort by similarity descending
    ranked_resumes = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    print(f"\nTop resumes for {job_id}:")
    for resume, score in ranked_resumes[:5]:  # Top 5 resumes
        print(f"{resume}: {score:.4f}")
     

In [ ]:
#Rank resumes for each job and store results
ranked_results = []

for i, job_id in enumerate(jobs_df['job_id']):
    sim_scores = list(zip(resumes_df['resume_id'], similarity_matrix[i]))
    # Sort by similarity descending
    ranked_resumes = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Save top N resumes (e.g., top 5)
    for rank, (resume_id, score) in enumerate(ranked_resumes[:5], start=1):
        ranked_results.append({
            "job_id": job_id,
            "resume_id": resume_id,
            "rank": rank,
            "similarity_score": score
        })

#Convert results to DataFrame
ranked_df = pd.DataFrame(ranked_results)

# Save ranked results to CSV
ranked_df.to_csv("../data/ranked_resumes.csv", index=False)

print("Ranking completed and saved to ../data/ranked_resumes.csv")
print(ranked_df.head(10))
